# 第 6 周练习：特定类别的微调

**目标：** 微调 GPT-4.1-nano，专门按类别（电子、玩具、汽车等）对产品进行定价，并将专业模型与通用模型进行比较。

**实验：**
1.加载数据集并按类别对项目进行分组
2. 在混合类别上微调**通用模型**（基线）
3. 使用足够的数据对 2-3 个类别的**特定类别模型**进行微调
4. 通过可视化评估每个类别的专家模型与一般模型

In [ ]:
# 进口
import os
from pathlib import Path
import json
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# 环境
load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

# 微调需要 OpenAI API。职位信息显示在 https://platform.openai.com/finetune
# OpenRouter 不支持微调（仅推理）。
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 可选：OpenRouter 客户端仅用于基本模型推理（不适用于微调模型）。
# 使用它进行零样本比较。微调模型 (ft:...) 仅适用于 OpenAI。
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
openrouter_client = (
    OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    if OPENROUTER_API_KEY
    else None
)

# 加载数据 - 使用 items_lite （具有 LLM 生成的摘要）进行微调
username = "ed-donner"
dataset = f"{username}/items_lite"
train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

# 按类别分组
def by_category(items):
    d = defaultdict(list)
    for item in items:
        d[item.category].append(item)
    return dict(d)

train_by_cat = by_category(train)
val_by_cat = by_category(val)
test_by_cat = by_category(test)

# 检查每个类别的计数
for cat in sorted(train_by_cat.keys()):
    print(f"{cat}: train={len(train_by_cat[cat])}, val={len(val_by_cat[cat])}, test={len(test_by_cat[cat])}")


## 1. 视觉：每个类别的项目

In [ ]:
# 每个类别的培训项目条形图
categories = sorted(train_by_cat.keys())
counts = [len(train_by_cat[c]) for c in categories]

plt.figure(figsize=(12, 5))
plt.bar(categories, counts, color="steelblue")
plt.title("Training Items per Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts):
    plt.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 2. 选择类别并准备微调数据

我们选择至少有 50 个训练示例和 10 个验证示例的类别。每个模型使用 100 个训练/25 个值。

In [ ]:
MIN_TRAIN = 50
MIN_VAL = 10
TRAIN_SIZE = 100
VAL_SIZE = 25

# 选择有足够数据的类别
selected_categories = [
    cat for cat in sorted(train_by_cat.keys())
    if len(train_by_cat[cat]) >= MIN_TRAIN and len(val_by_cat[cat]) >= MIN_VAL
]
print(f"Selected categories: {selected_categories}")

In [ ]:
# JSONL 帮助程序（与 day5 格式相同）
def messages_for(item, category=None):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation"
    if category:
        prompt = f"Estimate the price of this {category.replace('_', ' ')} product. Respond with the price, no explanation"
    return [
        {"role": "user", "content": f"{prompt}\n\n{item.summary}"},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items, category=None):
    result = ""
    for item in items:
        msgs = messages_for(item, category)
        result += '{"messages": ' + json.dumps(msgs) + "}\n"
    return result.strip()

def write_jsonl(items, filename, category=None):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(items, category))

## 3. 创建 JSONL 文件

- **通用模型：** 100 个来自所有类别的混合项目
- **类别专家：** 电子产品、玩具和游戏等各 100 件商品。

In [ ]:
import random
random.seed(42)

# 一般：从所有类别中抽取 100 个样本
general_train = []
per_cat = TRAIN_SIZE // len(selected_categories)
for cat in selected_categories:
    general_train.extend(random.sample(train_by_cat[cat], min(per_cat, len(train_by_cat[cat]))))
random.shuffle(general_train)
general_train = general_train[:TRAIN_SIZE]
general_val = val[:VAL_SIZE]

write_jsonl(general_train, "jsonl/general_train.jsonl")
write_jsonl(general_val, "jsonl/general_val.jsonl")
print(f"General: {len(general_train)} train, {len(general_val)} val")

In [ ]:
# 品类专家 - 使用前 2-3 个品类进行成本控制
SPECIALIST_CATEGORIES = selected_categories[:2]
print(f"Fine-tuning specialists for: {SPECIALIST_CATEGORIES}")

for cat in SPECIALIST_CATEGORIES:
    ct = train_by_cat[cat][:TRAIN_SIZE]
    cv = val_by_cat[cat][:VAL_SIZE]
    write_jsonl(ct, f"jsonl/{cat}_train.jsonl", category=cat)
    write_jsonl(cv, f"jsonl/{cat}_val.jsonl", category=cat)
    print(f"  {cat}: {len(ct)} train, {len(cv)} val")

## 4. 上传并开始微调作业

In [ ]:
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
HYPERPARAMS = {"n_epochs": 1, "batch_size": 1}

# 上传一般
with open("jsonl/general_train.jsonl", "rb") as f:
    general_train_file = client.files.create(file=f, purpose="fine-tune")
with open("jsonl/general_val.jsonl", "rb") as f:
    general_val_file = client.files.create(file=f, purpose="fine-tune")

# 开始一般微调
general_job = client.fine_tuning.jobs.create(
    training_file=general_train_file.id,
    validation_file=general_val_file.id,
    model=BASE_MODEL,
    seed=42,
    hyperparameters=HYPERPARAMS,
    suffix="pricer-general",
)
print(f"General job: {general_job.id}")

In [ ]:
# 上传并启动类别专家职位
specialist_jobs = {}
for cat in SPECIALIST_CATEGORIES:
    with open(f"jsonl/{cat}_train.jsonl", "rb") as f:
        tf = client.files.create(file=f, purpose="fine-tune")
    with open(f"jsonl/{cat}_val.jsonl", "rb") as f:
        vf = client.files.create(file=f, purpose="fine-tune")
    job = client.fine_tuning.jobs.create(
        training_file=tf.id,
        validation_file=vf.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters=HYPERPARAMS,
        suffix=f"pricer-{cat.lower()[:20]}",
    )
    specialist_jobs[cat] = job
    print(f"{cat}: {job.id}")

## 5.等待作业完成

In [ ]:
import time

def poll_until_done(job_id):
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        status = j.status
        if status == "succeeded":
            return j.fine_tuned_model
        if status == "failed":
            raise RuntimeError(f"Job {job_id} failed")
        print(f"  {job_id}: {status}")
        time.sleep(30)

# 获取通用模型
general_model = poll_until_done(general_job.id)
print(f"General model: {general_model}")

# 获取专业模型
specialist_models = {}
for cat, job in specialist_jobs.items():
    specialist_models[cat] = poll_until_done(job.id)
    print(f"{cat}: {specialist_models[cat]}")

## 6. 评估：专家与普通

- **通用型号：**用于所有项目
- **专家模型：**当物品类别匹配时使用；否则回退到一般

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def test_messages_for(item):
    return [{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}]

def predict(item, model_name):
    r = client.chat.completions.create(
        model=model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return r.choices[0].message.content

def general_pricer(item):
    return predict(item, general_model)

def specialist_router_pricer(item):
    """Use specialist if available, else general."""
    model = specialist_models.get(item.category, general_model)
    return predict(item, model)

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import re
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

def post_process(value):
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", value)
        return float(m.group()) if m else 0
    return float(value)

def mae_on_items(predictor, items, workers=5):
    errors = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for guess, item in zip(tqdm(ex.map(predictor, items), total=len(items), desc="Predicting"), items):
            g = post_process(guess)
            errors.append(abs(g - item.price))
    return sum(errors) / len(errors) if errors else 0

In [ ]:
# 按类别 MAE：普通与专业
# 每个类别使用较小的评估大小以提高速度（例如 50）
EVAL_SIZE = 50
results = []

for cat in SPECIALIST_CATEGORIES:
    test_items = test_by_cat.get(cat, [])[:EVAL_SIZE]
    if not test_items:
        continue
    mae_gen = mae_on_items(general_pricer, test_items)
    specialist_model = specialist_models.get(cat)
    mae_spec = mae_on_items(lambda i: predict(i, specialist_model), test_items) if specialist_model else None
    results.append({
        "category": cat,
        "n": len(test_items),
        "MAE (general)": mae_gen,
        "MAE (specialist)": mae_spec,
        "improvement": (mae_gen - mae_spec) if mae_spec else 0,
    })

results_df = pd.DataFrame(results)
results_df

## 7. 视觉：按类别划分的通用 MAE 与专业 MAE

In [ ]:
# 比较每个类别的普通 MAE 与专业 MAE 的条形图
df = results_df
x = range(len(df))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w/2 for i in x], df["MAE (general)"], w, label="General", color="steelblue")
ax.bar([i + w/2 for i in x], df["MAE (specialist)"], w, label="Specialist", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(df["category"], rotation=45, ha="right")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Category-Specific Fine-Tuning: General vs Specialist MAE")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 完整测试集评估（可选）

使用专业路由器在完整的测试集上运行“evaluate()”。

In [ ]:
# 在 200 个测试项目上评估专业路由器（使用 API 调用）
evaluate(specialist_router_pricer, test, size=200)

## 报告摘要

- **通用模型：** 在混合类别上进行训练；比较的基线。
- **专业模型：** 每个都接受单一类别的培训；预计在该类别上表现更好。
- **调查结果：** 检查条形图。积极的改进=该类别的专家胜过一般。